In [2]:
import pandas as pd
import numpy as np
import requests                  # for calling APIs and downloading websites
import yaml                      # for loading credentials
from bs4 import BeautifulSoup    # for parsing html
import time                      # to put the system to sleep
import random                    # for random numbers
import re                        # regular expressions

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings("ignore")

In [9]:
# Required to be able to access a wikipedia page
headers = {"User-Agent": "PPOL5203 class project"}

# step 1: find a website with information you want to collect
url = "https://en.wikipedia.org/wiki/ChatGPT"

# step 2: understand the website

# step 3: write code to collect one realization of the data
page = requests.get(url, headers=headers)
soup = BeautifulSoup(page.content, 'html.parser')

# grab release and strip it 
release = soup.select('th.infobox-label:-soup-contains("Release") + td')
cleaned = [i.get_text() for i in release]
cleaned[0].strip()

text = "\n".join([i.strip() for i in cleaned])
print(text[0:1000])


November 30, 2022(3 years ago) (2022-11-30)[1]


In [10]:
# helper function 
def get_infobox_value(soup, label):
    """helper function that collects the value associated with the label

    Args:
        soup: parsed HTML of a Wikipedia page.
        label: infobox label to look for: "Developer", "Release"
    Returns:
        str: text of the matching value cell, stripped of extra spaces
        None: if the label is not on the page
    """
    label = soup.select(f'th.infobox-label:-soup-contains("{label}") + td')

    if len(label) > 0:
        return label[0].get_text().strip()
    return None

In [11]:
def scrape_wiki_tool(url=None):
    """Collect developer, release year, and type from an AI tool's Wikipedia page.
    
    Downloads the page, reads the infobox with get_infobox_value(), and pulls the
    four-digit year out of the release text with a regular expression.
    
    Args:
        url (str): full Wikipedia URL for one AI tool "https://en.wikipedia.org/wiki/ChatGPT".
    Returns:
        dict: one row of tool facts with keys
    """
    # get access to the website
    page = requests.get(url, headers = headers)
    soup = BeautifulSoup(page.content, 'html.parser')
    if page.status_code == 200:   
        developer = get_infobox_value(soup, "Developer")
        release = get_infobox_value(soup, "Release")
        tool_type = get_infobox_value(soup, "Type")
        
        year = re.search(r'\d{4}', str(release))
        release_year = int(year.group()) if year else None

        return {"url": url,
                "developer": developer,
                "release_raw": release,
                "release_year": release_year,
                "type": tool_type}

scrape_wiki_tool("https://en.wikipedia.org/wiki/ChatGPT")

{'url': 'https://en.wikipedia.org/wiki/ChatGPT',
 'developer': 'OpenAI',
 'release_raw': 'November\xa030, 2022(3 years ago)\xa0(2022-11-30)[1]',
 'release_year': 2022,
 'type': 'Chatbot, LLM, GPT'}

In [12]:
tool_urls = {
    "ChatGPT": "https://en.wikipedia.org/wiki/ChatGPT",
    "Google Gemini": "https://en.wikipedia.org/wiki/Google_Gemini",
    "DeepSeek": "https://en.wikipedia.org/wiki/DeepSeek",
    "Microsoft Copilot": "https://en.wikipedia.org/wiki/Microsoft_Copilot",
    "Grammarly": "https://en.wikipedia.org/wiki/Grammarly",
    "QuillBot": "https://en.wikipedia.org/wiki/QuillBot",
    "Perplexity": "https://en.wikipedia.org/wiki/Perplexity_AI",
    "Photomath": "https://en.wikipedia.org/wiki/Photomath",
    "Midjourney": "https://en.wikipedia.org/wiki/Midjourney",
    "Grok": "https://en.wikipedia.org/wiki/Grok_(chatbot)",
    "Claude": "https://en.wikipedia.org/wiki/Claude_(language_model)",
}

# loop through and collect
results = []
for tool, url in tool_urls.items():
    #scrape the content
    row = scrape_wiki_tool(url)
    if row is not None:
        row["tool"] = tool
        results.append(row)
    # put the system to sleep for a random draw of time
    time.sleep(random.uniform(2, 5))   

In [13]:
# step 5: save
tools_df = pd.DataFrame(results)
tools_df.to_csv("wiki_ai_tools.csv", index=False)
tools_df

,url,developer,release_raw,release_year,type,tool
0,https://en.wikipedia.org/wiki/ChatGPT,OpenAI,"November 30, 2022(3 years ago) (2022-11-30)[1]",2022.0,"Chatbot, LLM, GPT",ChatGPT
1,https://en.wikipedia.org/wiki/Google_Gemini,Google AIGoogle DeepMind,"March 21, 2023; 3 years ago (2023-03-21)",2023.0,Chatbot\nGenerative artificial intelligence\nV...,Google Gemini
2,https://en.wikipedia.org/wiki/DeepSeek,None,None,NaN,Private,DeepSeek
3,https://en.wikipedia.org/wiki/Microsoft_Copilot,Microsoft AI,"February 7, 2023; 3 years ago (2023-02-07)",2023.0,Chatbot,Microsoft Copilot
4,https://en.wikipedia.org/wiki/Grammarly,Superhuman Platform Inc.,"July 1, 2009; 17 years ago (2009-07-01)[3]",2009.0,"Online text editor, browser extension, and mob...",Grammarly
5,https://en.wikipedia.org/wiki/QuillBot,None,None,NaN,None,QuillBot
6,https://en.wikipedia.org/wiki/Perplexity_AI,Aravind Srinivas\nDenis Yarats\nJohnny Ho\nAnd...,"December 7, 2022; 3 years ago (2022-12-07)",2022.0,Privately held company,Perplexity
7,https://en.wikipedia.org/wiki/Photomath,Google,2014 (2014),2014.0,Education Technology,Photomath
8,https://en.wikipedia.org/wiki/Midjourney,"Midjourney, Inc.","July 12, 2022; 4 years ago (2022-07-12) (open ...",2022.0,Text-to-image model,Midjourney
9,https://en.wikipedia.org/wiki/Grok_(chatbot),SpaceX Super IntelligenceCursor,"November 3, 2023; 3 years ago (2023-11-03)[1]",2023.0,Large language model\nGPT\nFoundation model\nC...,Grok
